# Real-Time Fatigue Detection - Model Evaluation
Evaluates the 1D-CNN fatigue detection model using standard classification metrics, RMSE, and visualizes the Confusion Matrix, ROC Curve, and Precision-Recall Curve.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, roc_curve, auc, precision_recall_curve
)

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

ModuleNotFoundError: No module named 'numpy'

### 1. Load Data & Synthesize Fatigue
This uses the **updated parameters** (`body_acc` instead of `total_acc`, amplified spikes, and dynamic activities list) to exactly match how the model was retrained.

In [ ]:
DATA_DIR = Path("data")
MODEL_PATH = Path("model/fatigue_model.keras")

SIGNAL_FILES = [
    "body_acc_x_{}.txt",
    "body_acc_y_{}.txt",
    "body_acc_z_{}.txt",
    "body_gyro_x_{}.txt",
    "body_gyro_y_{}.txt",
    "body_gyro_z_{}.txt",
]

def load_inertial_signals(split):
    signals = []
    for sig in SIGNAL_FILES:
        filepath = DATA_DIR / split / "Inertial Signals" / sig.format(split)
        signals.append(np.loadtxt(filepath))
    return np.stack(signals, axis=-1)

def load_labels(split):
    return np.loadtxt(DATA_DIR / split / f"y_{split}.txt", dtype=int)

def synthesize_fatigue(X_clean, rng):
    X_fat = X_clean.copy()
    n, steps, ch = X_fat.shape
    X_fat += rng.normal(0, 0.1, X_fat.shape) # Updated noise scale
    for i in range(n):
        for c in range(3):
            sig = X_fat[i, :, c]
            thresh = np.percentile(np.abs(sig), 90)
            mask = np.abs(sig) >= thresh
            sig[mask] *= rng.uniform(2.0, 4.0, size=np.sum(mask)) # Updated spike multiplier
            X_fat[i, :, c] = sig
    return X_fat

In [ ]:
rng = np.random.default_rng(seed=42)

print("Loading data...")
X_all = np.concatenate([load_inertial_signals("train"), load_inertial_signals("test")])
y_all = np.concatenate([load_labels("train"), load_labels("test")])

# Dynamic Activities: 1=WALKING, 2=WALKING_UPSTAIRS, 3=WALKING_DOWNSTAIRS
X_class0 = X_all[np.isin(y_all, [1, 2, 3])]
print(f"Optimal Class 0 Samples: {len(X_class0)}")

print("Synthesizing Fatigue Data...")
X_class1 = synthesize_fatigue(X_class0, rng)

X = np.concatenate([X_class0, X_class1])
y = np.concatenate([np.zeros(len(X_class0)), np.ones(len(X_class1))])

idx = rng.permutation(len(X))
X, y = X[idx], y[idx]

split = int(0.8 * len(X))
X_val, y_val = X[split:], y[split:]
print(f"Validation Samples: {len(X_val)}")

### 2. Load Model & Generate Predictions

In [ ]:
print("Loading model...")
model = tf.keras.models.load_model(MODEL_PATH)

print("Running predictions on validation set...")
y_prob = model.predict(X_val, verbose=0).flatten()
y_pred = (y_prob > 0.6).astype(int)
y_true = y_val.astype(int)

### 3. Compute Metrics (RMSE, Accuracy, F1, Precision)

In [ ]:
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_prob)) # RMSE calculation

print("=== PERFORMANCE METRICS ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"RMSE     : {rmse:.4f}")
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["Optimal (0)", "Fatigued (1)"]))

### 4. Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Optimal", "Fatigued"], 
            yticklabels=["Optimal", "Fatigued"])
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.show()

### 5. Advanced Evaluation Measures (ROC & Precision-Recall Curves)

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_prob)
pr_auc = auc(recall_vals, precision_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('Receiver Operating Characteristic (ROC)')
ax1.legend(loc="lower right")

# Precision-Recall Curve
ax2.plot(recall_vals, precision_vals, color='blue', lw=2, label=f'PR curve (AUC = {pr_auc:.4f})')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend(loc="lower left")

plt.tight_layout()
plt.show()